# DataFrame Optimised — Naive Bayes Classifier

**DataFrame implementation with three targeted optimisations over the baseline.**

All logic is identical to `naive_bayes_df_baseline.ipynb` except for
these three changes (each marked with `# OPTIMISATION N`):

1. Replace UDFs with built-in Spark functions (`F.concat`, `F.log`, etc.)
   so Catalyst can inspect and optimise the full query plan.
2. Call `.persist()` on the feature-counts DataFrame after training
   to avoid recomputing the shuffle if referenced again.
3. Call `.repartition(n)` at data load time to tune parallelism for the cluster.

These optimisations are symmetric to the RDD optimisations, making the
RDD-vs-DataFrame comparison apples-to-apples.

In [8]:
import time
import math
import sys
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

sys.path.insert(0, os.path.abspath('..'))

from core.naive_bayes import compute_log_probs, evaluate
from data.loader import load_car_dataframe, FEATURE_COLS, LABEL_COL

def predict(log_prob_table, test_point):
    best_class = None
    best_score = float('-inf')
    for class_label in log_prob_table['classes']:
        score = log_prob_table['log_class_probs'][class_label]
        for feat_idx, value in enumerate(test_point):
            key = f'feat_{feat_idx}_{value}_{class_label}'
            if key in log_prob_table['log_feature_probs']:
                score += log_prob_table['log_feature_probs'][key]
            else:
                score += log_prob_table['fallback_log_probs'].get(
                    (feat_idx, class_label), math.log(1e-10)
                )
        if score > best_score:
            best_score = score
            best_class = class_label
    return best_class

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('NaiveBayes-DF-Optimised')
    .getOrCreate()
)

In [10]:
DATA_PATH = '/Users/luccagazotto/Documents/École Polytechnique/DSAIB/Database Management/final_project/Database-Management---Naive-Bayes/data/car.csv'
NUM_PARTITIONS  = 4  # this value can be tuned

train_df, test_df = load_car_dataframe(spark, filepath=DATA_PATH)

# repartion the dataframes
train_df = train_df.repartition(NUM_PARTITIONS)
test_df  = test_df.repartition(NUM_PARTITIONS)

print(f'Train partitions: {train_df.rdd.getNumPartitions()}')

train_df.groupBy(LABEL_COL).count().orderBy('count', ascending=False).show()

Train partitions: 4
+-----+-----+
|label|count|
+-----+-----+
|unacc| 1008|
|  acc|  314|
| good|   58|
|vgood|   51|
+-----+-----+



26/04/02 20:17:55 WARN CacheManager: Asked to cache already cached data.


In [12]:
# training 

t_train_start = time.time()

class_counts_df = train_df.groupBy(LABEL_COL).agg(F.count('*').alias('class_count'))

# using concat to optimize the key generation for the feature counts
key_dfs = []
for feat_idx, col_name in enumerate(FEATURE_COLS):
    key_df = train_df.select(
        F.concat(
            F.lit(f'feat_{feat_idx}_'),
            F.col(col_name),
            F.lit('_'),
            F.col(LABEL_COL)
        ).alias('key')
    )
    key_dfs.append(key_df)

all_keys_df = key_dfs[0]
for df in key_dfs[1:]:
    all_keys_df = all_keys_df.union(df)

feature_counts_df = all_keys_df.groupBy('key').agg(F.count('*').alias('count'))

# using persist to cache the feature counts in memory 
feature_counts_df.persist()

train_time = time.time() - t_train_start
print(f'Training: {train_time:.4f}s')
print(f'Unique feature keys: {feature_counts_df.count()}')

Training: 0.0766s
Unique feature keys: 69


26/04/02 20:20:15 WARN CacheManager: Asked to cache already cached data.


In [13]:
# prob computation

class_counts_dict   = {row[LABEL_COL]: row['class_count'] for row in class_counts_df.collect()}
feature_counts_dict = {row['key']: row['count'] for row in feature_counts_df.collect()}
class_totals        = class_counts_dict

log_prob_table = compute_log_probs(class_counts_dict, feature_counts_dict, class_totals)

print(f"Classes           : {log_prob_table['classes']}")
print(f"Log class probs   : {log_prob_table['log_class_probs']}")
print(f"Sample feat probs : {list(log_prob_table['log_feature_probs'].items())[:3]}")

Classes           : ['unacc', 'acc', 'vgood', 'good']
Log class probs   : {'unacc': -0.35220510784011255, 'acc': -1.5163474893680884, 'vgood': -3.317676409612294, 'good': -3.191382684288002}
Sample feat probs : [('feat_3_2_unacc', -0.757487897325395), ('feat_4_med_unacc', -1.1226374682550995), ('feat_0_vhigh_acc', -1.6189166563886441)]


In [5]:
# Cell 5 — Prediction
#
# The predict UDF is still needed here — argmax over classes requires Python
# procedural logic that cannot be expressed as a Spark SQL expression.
# The optimisation over baseline is that we broadcast log_prob_table so it
# is sent once per executor instead of once per task.
bc_log_prob_table = sc.broadcast(log_prob_table)

# The UDF now accesses the broadcast value, not the driver-scope dict.
# Spark ships the broadcast variable once per executor; all tasks on that
# executor share the single cached copy in memory.
predict_udf = F.udf(
    lambda *features: predict(bc_log_prob_table.value, list(features)),
    StringType()
)

t_pred_start = time.time()

# Schema before: [buying, maint, doors, persons, lug_boot, safety, label]
# Schema after : [..., prediction]
prediction_df = test_df.withColumn(
    'prediction',
    predict_udf(*[F.col(c) for c in FEATURE_COLS])
)

results = prediction_df.select(LABEL_COL, 'prediction').collect()

pred_time = time.time() - t_pred_start
print(f'[TIMING] Prediction: {pred_time:.4f}s')
print(f'Sample: {results[:5]}')

[TIMING] Prediction: 0.5688s
Sample: [Row(label='unacc', prediction='unacc'), Row(label='unacc', prediction='unacc'), Row(label='unacc', prediction='unacc'), Row(label='acc', prediction='unacc'), Row(label='unacc', prediction='unacc')]


In [6]:
# Cell 6 — Evaluation

true_labels = [row[LABEL_COL] for row in results]
pred_labels = [row['prediction'] for row in results]

accuracy = evaluate(pred_labels, true_labels)

print(f'Accuracy     : {accuracy:.4f} ({accuracy * 100:.2f}%)')
print(f'Train time   : {train_time:.4f}s')
print(f'Predict time : {pred_time:.4f}s')
print(f'Total time   : {train_time + pred_time:.4f}s')

bc_log_prob_table.unpersist()
feature_counts_df.unpersist()

# TODO: Expected accuracy on the full car evaluation dataset is ~87.39%
# (Zheng 2014). With the 5-row dummy dataset accuracy is not meaningful.
# Replace DATA_PATH with the real file path before interpreting results.

Accuracy     : 0.8552 (85.52%)
Train time   : 0.1059s
Predict time : 0.5688s
Total time   : 0.6747s


DataFrame[key: string, count: bigint]